In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# Phrase A
# 3.2
data = pd.read_csv(r"data.csv", encoding="latin-1")
data["CustomerID"] = data["CustomerID"].astype(dtype="Int64")
print("data head:\n", data.head())
print("\n\n")
print("data info:\n", data.info())
print("\n\n")


In [ ]:
# 3.9
cols = ["Country", "StockCode", "Description"]

print("Number of Unique Values:")
print(data[cols].nunique())

for col in cols:
  print(f"\n{'-' * 25}\n")
  print(data[col].value_counts(dropna=False).head())

print(f"\n{'-' * 25}\n")
print("Non-product StockCodes:")
stock_code = data["StockCode"]
# The valid StockCode begins with a 5-digit number.
is_product = data["StockCode"].astype(str).str.match(r"^\d{5}")
non_product = stock_code[~is_product]
print(non_product.value_counts().head())

In [ ]:
# 3.8
cols = ["Quantity", "UnitPrice"]

print("Summary of Quantity and UnitPrice:")
print(data[cols].describe())

# Negative Quantity
print(f"\n{'-' * 25}\n")
invalid_quantity = data.loc[data["Quantity"] < 0]
print(f"Invalid quantity: {len(invalid_quantity)}")
# print(invalid_quantity.head())

# Zero/negative UnitPrice
print(f"\n{'-' * 25}\n")
invalid_price = data.loc[data["UnitPrice"] <= 0]
print(f"Invalid prices (invalid sales): {len(invalid_price)}")
print(f"\tUnitPrice == 0: {(data['UnitPrice'] == 0).sum()}")
print(f"\tUnitPrice <  0: {(data['UnitPrice'] < 0).sum()}")
# print(invalid_price.head()

In [ ]:
# 3.1
COUNTRY_TO_REGION = {
    # UK & Ireland
    "United Kingdom": "UK&IE",
    "EIRE": "UK&IE",  # Ireland
    "Channel Islands": "UK&IE",
    # Western Europe
    "France": "Western Europe",
    "Germany": "Western Europe",
    "Belgium": "Western Europe",
    "Netherlands": "Western Europe",
    "Austria": "Western Europe",
    "Switzerland": "Western Europe",
    # Northern Europe
    "Denmark": "Northern Europe",
    "Finland": "Northern Europe",
    "Norway": "Northern Europe",
    "Sweden": "Northern Europe",
    "Iceland": "Northern Europe",
    # Southern Europe
    "Italy": "Southern Europe",
    "Spain": "Southern Europe",
    "Portugal": "Southern Europe",
    "Greece": "Southern Europe",
    "Malta": "Southern Europe",
    "Cyprus": "Southern Europe",
    # Eastern Europe
    "Poland": "Eastern Europe",
    "Czech Republic": "Eastern Europe",
    "Lithuania": "Eastern Europe",
    # North America
    "USA": "North America",
    "Canada": "North America",
    # Middle East
    "Bahrain": "Middle East",
    "Israel": "Middle East",
    "Lebanon": "Middle East",
    "Saudi Arabia": "Middle East",
    "United Arab Emirates": "Middle East",
    # Asia-Pacific
    "Hong Kong": "Asia-Pacific",
    "Japan": "Asia-Pacific",
    "Singapore": "Asia-Pacific",
    "Australia": "Asia-Pacific",
    # Other
    "Brazil": "South America",
    "RSA": "Africa",  # South Africa
    "European Community": "Unknown",
    "Unspecified": "Unknown",
}

# Check
for ct in data["Country"].unique():
    if ct not in COUNTRY_TO_REGION:
        print(f"Country '{ct}' is not in the mapping dictionary.")
else:
    print("All countries are accounted for in the mapping dictionary.")


In [ ]:
# Phase B
# 3.3
# print(data.iloc[0:5, 0:3])

# Create a unique line-item index for the DataFrame (format: "L000000", "L000001", etc.)
data_num = len(data)
data.index = pd.Index([f"L{i:06d}" for i in range(data_num)], name="line_item")
# print(data.loc["L000005"])
data

In [ ]:
# 3.4
is_cancelled = data["InvoiceNo"].astype(str).str.startswith("C")
bad_qty = data["Quantity"] <= 0
bad_price = data["UnitPrice"] <= 0

print(f"Number of canceled orders: {is_cancelled.sum()}")
print(f"Number of orders with non-positive quantity: {bad_qty.sum()}")
print(f"Number of orders with non-positive unit price: {bad_price.sum()}")

In [ ]:
# 3.5
data["Revenue"] = data["Quantity"] * data["UnitPrice"]
data["Revenue"] = data["Revenue"].round(3)
cols = ["InvoiceNo", "Quantity", "UnitPrice", "Revenue"]
print("Top largest bulk orders:")
print(data.sort_values("Quantity", ascending=False)[cols].head())
print("\nTop largest returns:")
print(data.sort_values("Revenue", ascending=True)[cols].head())

In [ ]:
# Phase C
# 3.6
data["Country"] = data["Country"].replace({ "EIRE": "Ireland", "RSA": "South Africa", "Unspecified": pd.NA })  # type: ignore
print("EIRE -> Ireland, RSA -> South Africa, Unspecified -> missing")
print(data["Country"].value_counts(dropna=False).head())

In [ ]:
# 3.7
COLUMN_MAP={
    "InvoiceNo": "invoice_no", "StockCode": "stock_code",
    "Description": "description", "Quantity": "quantity",
    "InvoiceDate": "invoice_date", "UnitPrice": "unit_price",
    "CustomerID": "customer_id", "Country": "country",
}
data.rename(columns=COLUMN_MAP, inplace=True)
print(data.columns)


In [ ]:
# 3.10
# Custom ID
print("Number of missing customer IDs:", data["customer_id"].isnull().sum())
data["customer_id"] = data["customer_id"].fillna(-1)
# Empty descriptions
# mask = data["description"].str.strip().eq("")
print("Number of empty descriptions:", data["description"].isnull().sum())
data["description"] = data["description"].fillna(pd.NA)


In [ ]:
# 3.11
# [TODO]?
# data.dropna(subset=["description"])

In [ ]:
# 3.12
# is_cancelled bad_price bad_qty
print(f"Data size before dropping invalid rows: {len(data)}")
data = data.drop(index=data[is_cancelled | bad_price | bad_qty].index)
not_product = (~data["stock_code"].astype(str).str.match(r"^\d{5}"))
data = data.drop(index=data[not_product].index)
data = data.dropna(subset=["description"])
print(f"Data size after dropping invalid rows: {len(data)}")

In [ ]:
# 3.13
duplicate_number = int(data.duplicated().sum())
data = data.drop_duplicates().reset_index(drop=True)
print("Exact duplicate rows removed:", duplicate_number)
print("Rows remaining:", len(data))

In [ ]:
# Phase D
# 3.18
data.rename(columns={"Revenue": "revenue"}, inplace=True)
# data.columns

data["description"] = data["description"].apply(lambda s: str(s).strip().title())
data["is_cancelled"] = data["invoice_no"].astype(str).str.startswith("C")

data.head()

In [ ]:
# 3.17
import time

s = time.time()
cleaned_desc_list = []
unique_data = data["description"].unique()
for des in unique_data:
    cleaned = str(des).strip().title()
    cleaned_desc_list.append(cleaned)
e = time.time()
print(f"for-loop: time cost in {e - s:.4f} seconds")

s = time.time()
cleaned_desc_list = [
    str(des).strip().title() for des in data["description"].unique()
]
e = time.time()
print(f"list comprehension: time cost in {e - s:.4f} seconds")

s = time.time()
data["description"] = data["description"].apply(lambda s: str(s).strip().title())
e = time.time()
print(f"apply: time cost in {e - s:.4f} seconds")

# In my timing test, approach `list comprehension` > `for-loop` > `apply`, in terms of speed.
# This might be reasonable because apply often still executes Python-level operations element by element.

In [ ]:
# 3.14
rev_by_country = data.groupby("country")["revenue"].sum().sort_values(ascending=False)
print("Revenue by country:", rev_by_country.head(), sep="\n")

print(f"\n{'-' * 25}\n")

orders_per_customer = data[data["customer_id"] != -1].groupby("customer_id")["invoice_no"].nunique().sort_values(ascending=False)
print("Orders per customer:", orders_per_customer.head(), sep="\n")

In [ ]:
# 3.16
by_country = data.groupby("country").agg(
    total_revenue=("revenue", "sum"),
    mean_line_value=("revenue", "mean"),
    transactions=("invoice_no", "nunique"),
).sort_values("total_revenue", ascending=False)
# by_country.head()

mask = data["customer_id"] != -1
by_customer = data[mask].groupby("customer_id").agg(
    total_revenue=("revenue", "sum"),
    mean_line_value=("revenue", "mean"),
    transactions=("invoice_no", "nunique"),
).sort_values("transactions", ascending=False)
by_customer.head()

In [ ]:
# 3.19
data["invoice_date"] = pd.to_datetime(data["invoice_date"])  # Also for 3.15

valid_customers = data[data["customer_id"] != -1]

def customer_summary(cs):
    return pd.Series({
        "total_spend":   cs["revenue"].sum(),
        "n_orders":      cs["invoice_no"].nunique(),
        "active_months": cs["invoice_date"].dt.to_period("M").nunique(),
    })

per_customer = valid_customers.groupby("customer_id").apply(customer_summary)
per_customer["n_orders"] = per_customer["n_orders"].astype(int)
per_customer["active_months"] = per_customer["active_months"].astype(int)
per_customer.sort_values("n_orders", ascending=False).head()


In [ ]:
# 3.15
ts = data.set_index("invoice_date").sort_index()

# Monthly
monthly = ts["revenue"].resample("MS").sum()
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(monthly.index, monthly.values, marker="o")
ax.set_title("Monthly revenue"); ax.set_xlabel("Month"); ax.set_ylabel("Revenue")
fig.autofmt_xdate(); fig.tight_layout()
fig.savefig(r"monthly_revenue.png")

# Weekly
weekly = ts["revenue"].resample("W").sum()
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(weekly.index, weekly.values, marker="o")
ax.set_title("Weekly revenue"); ax.set_xlabel("Week"); ax.set_ylabel("Revenue")
fig.autofmt_xdate(); fig.tight_layout()
fig.savefig(r"weekly_revenue.png")
plt.show()

In [ ]:
# 3.20

In [ ]:
# 3.21

In [ ]:
data.to_csv(r"cleaned_online_retail.csv", index=False)
data.dtypes